<a href="https://colab.research.google.com/github/RodionOm/Search-ranking-ml/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RodionOm/Search-ranking-ml/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Chosen lane: Lane 2 — Refresh / Content Opportunity Scoring.**

I pick this lane because the question it answers — *which pages should a reviewer look at first, given limited capacity* — is a concrete decision-support problem, and the starter pipeline already shows that a learned ranking beats a transparent rule on it (Precision@50: 0.240 for the hand-written rule vs 0.740 for the random forest, under client-holdout validation). That gives me a verified baseline to improve on rather than a blank page, and it reuses a tabular XGBoost + SHAP workflow I already know.

**Planned refinement (Week 3+):** the starter label `is_declining_label = (trend_direction == "down")` is a proxy computed from the current window, not a future outcome. For the capstone I plan to move to a future-looking label on the warehouse daily facts — features from a prior 90-day window predicting decline over the next 30 days — with a strict leakage audit. For this framing assignment I stay on the starter dataset.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

## 2. The question: decision, action, cost of a wrong call

**Research question:** Using observable search and content signals available *before* any editorial decision, which pages should be prioritised for refresh review first?

**Unit of analysis (grain):** one content page (one row = one `content_id`).

**Output:** a ranked review queue — pages ordered by refresh priority, each with a score and human-readable reason codes.

**Decision it improves:** how the content team allocates its limited weekly review capacity across thousands of pages.

**Action someone takes:** a reviewer works down the top of the queue and decides, per page, whether to refresh / expand / protect / monitor — using the reason codes as context.

**Cost of a wrong call:** a false positive near the top of the queue spends a reviewer's scarce time on a page that did not need it, and displaces a page that did. Because capacity is the binding constraint, ranking errors at the top are far more costly than errors deep in the list — which is exactly why the metric is Precision@K, not overall accuracy.

**Why this is NOT just "train a model":** this is a prioritisation problem under a capacity limit, not a pure prediction problem. "The right page to fix" means "the right page to review first, given limited time" — not a page guaranteed to recover if edited. Proving that a refresh *causes* recovery would need an experiment this data cannot provide. So the deliverable is a decision-support ranking with inspectable reason codes.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

## 3. Quick look at the data (2-3 real numbers)

**What these numbers say:**

1. **54.2% of pages carry the decline proxy label** — the target is common enough to rank, not degenerate. With classes near 50/50, a naive guess sits around chance, so the value has to come from ordering the top of the queue well.
2. **search_volume barely correlates with real traffic (0.001)** — the obvious "prioritise high-volume keywords" heuristic fails.
3. **word_count is nearly identical for declining vs growing pages (2909 vs 2848)** — another intuitive single-signal rule fails.

Together: simple one-column rules are weak here, and the starter run already shows a learned ranking roughly tripling the rule's Precision@50 — why this lane is worth the next seven weeks.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# This cell is for CODE (numbers, a query, a check).

import os, sys, subprocess, pandas as pd

# --- ensure we can find the data (works in Colab and locally) ---
if "google.colab" in sys.modules and not os.path.isdir("data/raw"):
    if not os.path.isdir("flyrank-ml-internship-starter"):
        subprocess.run(["git","clone","--depth","1",
            "https://github.com/flyrank-bih/flyrank-ml-internship-starter",
            "flyrank-ml-internship-starter"], check=True)
    os.chdir("flyrank-ml-internship-starter")
while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Rows: {df.shape[0]:,}  Columns: {df.shape[1]}")

# 1. How common is the target — is there enough to rank?
decline_rate = (df["trend_direction"] == "down").mean()
print(f"1. Declining pages (proxy label): {decline_rate:.1%} of pages")

# 2. search_volume does NOT predict real traffic -> can't sort by it
corr = df["search_volume"].corr(df["impressions_90d"])
print(f"2. corr(search_volume, impressions_90d) = {corr:.3f}  (near zero)")

# 3. content length does NOT separate declining from growing pages
wc = df.groupby("trend_direction")["word_count"].median()
print(f"3. Median word_count — down: {wc.get('down', float('nan')):.0f}, "
      f"up: {wc.get('up', float('nan')):.0f}  (nearly identical)")

Rows: 30,000  Columns: 44
1. Declining pages (proxy label): 54.2% of pages
2. corr(search_volume, impressions_90d) = 0.001  (near zero)
3. Median word_count — down: 2909, up: 2848  (nearly identical)


## 4. Careful words: what I can and can't claim

**I can claim (observed / directional, in this dataset):**
- In this 30,000-row anonymized starter slice, a learned ranking achieved a higher Precision@50 than the transparent baseline rule.
- Certain observable signals are *associated* with the decline proxy label.
- Single-signal heuristics (search volume, word count) are weak predictors here.

**I cannot claim:**
- That I proved any Google ranking factor.
- That refreshing a page *causes* it to recover — this data has no experiment.
- That the starter result generalises to the full warehouse; it must be re-earned there with proper time-aware and grouped validation.
- Anything about a specific client, domain, URL, or query — all pseudonymized.

All outputs use pseudonymized IDs and aggregated, public-safe metrics only.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.